# CineData Analytics — Landing to Bronze

Este notebook é responsável pela ingestão dos dados brutos da camada Landing para a camada Bronze.

Nesta etapa:
- os arquivos CSV são lidos a partir de um Volume do Databricks;
- os dados são mantidos próximos à estrutura original;
- é adicionada a coluna `ingestion_datetime` para rastreabilidade;
- as tabelas são persistidas em formato Delta;
- os dados são inseridos utilizando o modo `append`;
- a cotação do dólar é obtida através da API do Banco Central.

In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql("USE CATALOG workspace")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

DataFrame[]

In [0]:
landing_path = "/Volumes/workspace/default/inputs_cinedata"

In [0]:
arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

In [0]:
for arquivo, tabela in arquivos_tabelas.items():

    caminho_arquivo = f"{landing_path}/{arquivo}"

    df_bronze = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(caminho_arquivo)
        .withColumn("ingestion_datetime", F.current_timestamp())
    )

    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela)
    )

    print(f"{arquivo} -> {tabela} carregada com sucesso.")

movies_info_TMDB_IMDB.csv -> bronze.tb_movies_info carregada com sucesso.
movies_financials_IMDB_TMDB.csv -> bronze.tb_movies_financials carregada com sucesso.
movies_metrics_IMDB_TMDB.csv -> bronze.tb_movies_metrics carregada com sucesso.
credits_and_tags_IMDB_TMDB.csv -> bronze.tb_credits_and_tags carregada com sucesso.
movies_reviews.csv -> bronze.tb_movies_reviews carregada com sucesso.


## Ingestão da cotação do dólar — API Banco Central

A cotação do dólar é obtida através da API PTAX do Banco Central do Brasil.

As datas de início e fim são parametrizadas por widgets, permitindo a reutilização do notebook em diferentes períodos de execução.

Como a API não retorna valores em finais de semana e feriados, é utilizada por padrão uma janela correspondente aos últimos 7 dias corridos.

In [0]:
from datetime import datetime, timedelta

In [0]:
data_fim_padrao = datetime.today()
data_inicio_padrao = data_fim_padrao - timedelta(days=7)

dbutils.widgets.text(
    "data_inicio",
    data_inicio_padrao.strftime("%m-%d-%Y"),
    "Data inicial"
)

dbutils.widgets.text(
    "data_fim",
    data_fim_padrao.strftime("%m-%d-%Y"),
    "Data final"
)

In [0]:
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Período consultado: {data_inicio} até {data_fim}")

Período consultado: 09-11-2026 até 09-18-2026


In [0]:
import requests

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

parametros = {
    "@dataInicial": f"'{data_inicio}'",
    "@dataFinalCotacao": f"'{data_fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

response = requests.get(
    url,
    params=parametros,
    timeout=30
)

response.raise_for_status()

dados_cotacao = response.json()["value"]

print(f"{len(dados_cotacao)} cotações encontradas.")

6 cotações encontradas.


In [0]:
df_cotacao_dolar = spark.createDataFrame(dados_cotacao)

display(df_cotacao_dolar)

cotacaoCompra,dataHoraCotacao
5.0912,2026-09-11 13:07:22.532196
5.169,2026-09-14 13:10:08.144425
5.1484,2026-09-15 13:09:19.199664
5.152,2026-09-16 13:05:30.35873
5.1515,2026-09-17 13:03:21.858212
5.1569,2026-09-18 13:03:34.742036


In [0]:
df_cotacao_dolar = (
    df_cotacao_dolar
    .withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )
)

In [0]:
(
    df_cotacao_dolar.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

print("Cotação do dólar -> bronze.tb_cotacao_dolar carregada com sucesso.")

Cotação do dólar -> bronze.tb_cotacao_dolar carregada com sucesso.


## Validação da camada Bronze

Validação das tabelas criadas e da quantidade de registros ingeridos.

In [0]:
tabelas_bronze = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

for tabela in tabelas_bronze:
    quantidade = spark.table(f"bronze.{tabela}").count()
    print(f"bronze.{tabela}: {quantidade} registros")

bronze.tb_movies_info: 106930 registros
bronze.tb_movies_financials: 106165 registros
bronze.tb_movies_metrics: 107364 registros
bronze.tb_credits_and_tags: 106320 registros
bronze.tb_movies_reviews: 32412 registros
bronze.tb_cotacao_dolar: 6 registros
